# Asynchrónne programovanie

## Čo je thread (vlákno)
**Vlákno** (`Thread`) = fyzický vykonávací kontext, v ktorom beží kód. Každé vlákno má svoj zásobník (stack) a operačný systém ho plánuje na jadro CPU.

Každý .NET proces má aspoň jedno vlákno – tzv. **hlavné vlákno** alebo **primárne vlákno**, ktoré beží od začiatku spustenia.

Id vlákna môžeme vypísať pomocou vlastnosti `ManagedThreadId` (keď spustíte túto bunku viackrát, malo by byť ID rovnaké, až dokiaľ nereštartujete alebo znova nespustíte tento notebook):

In [ ]:
using System.Threading;

Console.WriteLine(Thread.CurrentThread.ManagedThreadId);

Môžeme vytvárať viacero vlákien (spustite túto bunku viackrát a sledujte ako sa menia vypísané ID vlákien):

In [ ]:
var thread1 = new Thread(() => Console.WriteLine(Thread.CurrentThread.ManagedThreadId));
thread1.Start();

var thread2 = new Thread(() => Console.WriteLine(Thread.CurrentThread.ManagedThreadId));
thread2.Start();

Dajme si konkrétnejší príklad - zavolanie funkcie `PrintNumbers` z 2 vlákien (hlavného a sekundárneho):

In [ ]:
void PrintNumbers(string name)
{
    for (int i = 1; i <= 5; i++)
    {
        Console.WriteLine($"{i} {name} (thread {Thread.CurrentThread.ManagedThreadId})");
        Thread.Sleep(Random.Shared.Next(50, 500));
    }
}

var thread = new Thread(() => PrintNumbers("Sekundárne vlákno"));
thread.Start();

PrintNumbers("Hlavné vlákno");

## Čo je Task?
**Task** = objektová reprezentácia práce, ktorá môže bežať na vlákne. `Task` je logická jednotka – môže bežať paralelne, sekvenčne alebo čakať na I/O

In [ ]:
var task = new Task(() =>
{
    Console.WriteLine($"Bežím v tasku {Task.CurrentId} (thread {Thread.CurrentThread.ManagedThreadId})"); // Vlákno sa použije z ThreadPool-u
});
task.Start();

Console.WriteLine($"Bežím v hlavnom vlákne (thread {Thread.CurrentThread.ManagedThreadId})");

Odštartovanie tasku môžeme zabezpečiť aj pomocou statickej metódy `Run`:

In [ ]:
var task = Task.Run(() =>
{
    Console.WriteLine($"Bežím v tasku {Task.CurrentId} (thread {Thread.CurrentThread.ManagedThreadId})");
});

Console.WriteLine($"Bežím v hlavnom vlákne (thread {Thread.CurrentThread.ManagedThreadId})");

Všimnite si poradie výpisu vyššie.

Ak chceme čakať na dokončenie tasku, použijeme kľúčové slovo `await`:

In [ ]:
Console.WriteLine($"Bežím v hlavnom vlákne {Thread.CurrentThread.ManagedThreadId}");

var task = Task.Run(() =>
{
    Console.WriteLine($"Bežím v tasku {Task.CurrentId} (thread {Thread.CurrentThread.ManagedThreadId})");
});
await task;

Console.WriteLine($"Bežím vo vlákne {Thread.CurrentThread.ManagedThreadId}"); // Id vlákna záleží od synchronizačného kontextu, či sa prepne do pôvodného vlákna alebo zostane (tu je iné, v UI by sa preplo do pôvodného - hlavného vlákna)

## Čo znamená úloha typu CPU-bound a I/O-bound?

### CPU-bound (výpočtové úlohy)
Spotrebúvajú procesor, nie sú viazané na čakanie. Príklady: výpočty, spracovanie obrázkov, kompresia, kryptografia. 

In [ ]:
using System.Diagnostics;

double Compute()
{
    double sum = 0;
    for (int i = 0; i < 50_000_000; i++)
        sum += Math.Sqrt(i);
    return sum;
}

var sw = Stopwatch.StartNew();
var result = Compute(); // Blokuje aktuálne vlákno - synchrónna metóda
sw.Stop();

Console.WriteLine($"Výsledok: {result}, trvalo {sw.ElapsedMilliseconds} ms");

Ak to chceme presunúť na pozadie:

In [ ]:
sw.Restart();
var result2 = await Task.Run(() => Compute()); // Beží na inom vlákne
sw.Stop();

Console.WriteLine($"Asynchrónne: {result2}, {sw.ElapsedMilliseconds} ms");

### I/O-bound (vstupno-výstupné úlohy)
Väčšinu času nevyužíva CPU, čaká na I/O (disk, sieť, databáza…). Asynchrónne API umožňuje, aby vlákno nebolo zablokované počas čakania.

In [ ]:
async Task DownloadDataAsync()
{
    Console.WriteLine($"Začínam sťahovanie (thread {Thread.CurrentThread.ManagedThreadId})");
    await Task.Delay(2000); // simuluje čakanie na sieť
    Console.WriteLine($"Dáta prijaté (thread {Thread.CurrentThread.ManagedThreadId})");
}

await DownloadDataAsync();

Počas `await Task.Delay` sa vlákno uvoľní pre inú prácu. Keď je I/O hotové, await pokračuje (možno na inom vlákne).

Príklad z .NET knižnice na uloženie textu do textového súboru:

In [ ]:
using System.IO;

const string file = "subor.txt";
string text = "Ahoj :)";

// Blokujúca - synchrónna verzia:
File.WriteAllText(file, text);

// Neblokujúca - asynchrónna verzia:
await File.WriteAllTextAsync(file, text);

Stiahnutie obsahu z internetu pomocou `HttpClient`:

In [ ]:
using System.Net.Http;

var client = new HttpClient();
var htmlContent = await client.GetStringAsync("https://www.fri.uniza.sk/");
Console.WriteLine("Hotovo, stránka stiahnutá");
client.Dispose();

await File.WriteAllTextAsync("fri-uniza-sk.html", htmlContent);
Console.WriteLine("Stránka uložená do súboru");

Ak by sme chceli sťahovať súčasne dve stránky naraz, nepoužijeme await hneď, ale až po vrátení taskov: 

In [ ]:
var client = new HttpClient();

var friUnizaDownloaderTask = client.GetStringAsync("https://www.fri.uniza.sk/");
var unizaDownloaderTask = client.GetStringAsync("https://www.uniza.sk/");

string friUnizaHtmlContent = await friUnizaDownloaderTask;
string unizaHtmlContent = await unizaDownloaderTask;

Console.WriteLine("Hotovo, obe stránky boli stiahnuté");
client.Dispose();

Každá úloha obsahuje veľa vlastností, z ktorých zaujímavé sú napr. `Result`, `Id`, `Status` alebo `Exception`:

In [ ]:
Console.WriteLine("Obsahy sú rovnaké? " + (unizaDownloaderTask.Result == unizaHtmlContent)); 
Console.WriteLine("IsCompleted? " + unizaDownloaderTask.IsCompleted); 
Console.WriteLine("Id = " + unizaDownloaderTask.Id);
Console.WriteLine("Status = " + unizaDownloaderTask.Status);
Console.WriteLine("Exception = " + unizaDownloaderTask.Exception);
Console.WriteLine("Result:");
unizaDownloaderTask.Result

Task obsahuje aj metódy `Task.WhenAll()` a `Task.WhenAny()`, ktoré vracajú task umožňujúci čakať na dokončenie všetkých alebo jednej z úloh: 

In [ ]:
var client = new HttpClient();

string[] urls = new[] { "https://www.fri.uniza.sk/", "https://www.uniza.sk/", "https://vzdelavanie.uniza.sk/" };

var tasks = new List<Task>();
for (int i = 0; i < urls.Length; i++)
{
    var url = urls[i];
    tasks.Add(Task.Run(async () =>
    {
        try
        {
            var content = await client.GetStringAsync(url);
            var host = new Uri(url).Host.Replace('.', '-');
            var fileName = $"{host}.html";
            await File.WriteAllTextAsync(fileName, content);
            Console.WriteLine($"Stiahnuté {url} -> {fileName}");
        }
        catch (Exception ex)
        {
            Console.WriteLine($"Chyba pri sťahovaní {url}: {ex.Message}");
        }
    }));
}

await Task.WhenAll(tasks); // Vyskúšajte si tu použiť namiesto WhenAll: await Task.WhenAny(tasks);  
client.Dispose();

## Úloha 1

Nasledujúca metóda počíta číslo Pi:

In [ ]:
using System;
using System.Diagnostics;
using System.Threading.Tasks;

double ComputePiNilakantha(long terms)
{
    double pi = 3.0;
    double sign = 1.0;
    for (long k = 1; k <= terms; k++)
    {
        double a = 2.0 * k;
        double term = 4.0 / (a * (a + 1.0) * (a + 2.0));
        pi += sign * term;
        sign = -sign;
    }
    return pi;
}

var pi = ComputePiNilakantha(terms: 500_000_000);
Console.WriteLine($"π ≈ {pi}");

Je toto I/O bound alebo CPU bound úloha? Zavolajte metódu `ComputePiNilakantha` pomocou tasku:

In [ ]:
// TODO: Zavolať metódu ComputePiNilakantha v tasku


## Úloha 2

Nasledujúca metóda je synchrónna:

In [ ]:
public string GetMessage()
{
    return "Ahoj svet!";
}

Ak ju zavoláme, dostaneme reťazec:

In [ ]:
var message = GetMessage();
Console.WriteLine(message);

Ak by sme ju chceli prerobiť na asynchrónnu, návratový typ bude `Task<string>`: 

In [ ]:
// TODO: Doprogramujte

Zavolajte metódu `GetMessageAsync` a vypíšte na obrazovku jej návratovú hodnotu:

In [ ]:
// V tejto metóde je chyba, vracia System.Threading.Tasks.Task`1[System.String] namiesto "Ahoj, async svet!" 
// TODO: Upravte volanie metódy tak, aby sa správne počkalo na výsledok asynchrónnej metódy.
var message = GetMessageAsync();
Console.WriteLine(message);

Nasledujúca metóda vypisuje znaky reťazca so spomalením:

In [ ]:
using System.Threading;

public void PrintMessage(string message)
{
    foreach (var ch in message)
    {
        Console.Write(ch);
        Thread.Sleep(500); // Simulujeme spomalenie cez Thread - tu "zamrzne" vlakno
    }

    Console.WriteLine();
}

PrintMessage("Ahoj");

Prerobte metódu `PrintMessage` na asynchrónnu s využitím metódy `Task.Delay()` namiesto `Thread.Sleep()`: 

In [ ]:
// TODO: Sem napíšte metódu PrintMessage


## Úloha 3

Niekedy potrebujeme čakať - vtedy je vhodné použiť `Task.Delay()` metódu:

In [ ]:
public async Task SimulateLongTaskAsync()
{
    Console.WriteLine("Začínam operáciu");
    await Task.Delay(3000);
    Console.WriteLine("Operácia dokončená");
}

await SimulateLongTaskAsync();

Napíšte kód, ktorý postupne vypíše s použitím `Task.Delay()` odpočet 3 2 1 Štart po jednej sekunde:
>3
>2
>1
>Štart

In [ ]:
// TODO: Napíšte kód


## Úloha 4

Vytvoríme dve asynchrónne metódy: jedna bude simulovať stiahnutie dát (2 sekundy) a druhá spracovanie dát (1 sekunda).

In [ ]:
public async Task DownloadDataAsync()
{
    // Simuluje stiahutie dát
    await Task.Delay(1000);
    Console.WriteLine("Dáta stiahnuté");
}

public async Task ProcessDataAsync()
{
    // Simuluje spracovanie dát
    await Task.Delay(1500);
    Console.WriteLine("Dáta spracované");
}

Ak chceme, aby bežali jedna úloha po druhej, môžeme použiť metódu `ContinueWith()` (všimnite si výstup a aj čas vykonania - 2,5 s):

In [ ]:
var task = DownloadDataAsync().ContinueWith(previousTask => ProcessDataAsync()).Unwrap(); // Skuste odstranit Unwrap a pozorujte rozdiel
await task;

Avšak, predchádzajúci prípad je dosť komplikovaný, preto v modernom C# použite radšej volania cez await:

In [ ]:
await DownloadDataAsync();
await ProcessDataAsync();

Ak nepoužijeme kľúčové slovo await, metódy sú spustené, ale nevieme, kedy skončia, ak si neuložíme návratovú hodnotu tasku:

In [ ]:
DownloadDataAsync(); // Tu sa zobrazuje warning, lebo si neukladáme návratovú hodnotu tasku
Console.WriteLine("Stiahnutie dokončené, začínam spracovanie..."); // Chyba, nevieme, kedy skončí stiahnutie
ProcessDataAsync();
Console.WriteLine("Spracovanie dokončené."); // Chyba, nevieme, kedy skončí spracovanie

// Počkáme 3 sekundy, aby sme videli výstup, keďže vieme, že spracovanie trvá 2,5 sekundy
await Task.Delay(3000);

Zavolajte metódu `DownloadDataAsync();` 10x súbežne a počkajte na dokončenie všetkých úloh (nech to netrvá 10 x 1 = 10 sekúnd, ale do 1-2 sekundy):

In [ ]:
// TODO: Vykonajte 10x DownloadDataAsync() paralelne


## Úloha 5

Stiahnite a uložte pomocou `HttpClient` nasledujúce jpg obrázky z https://picsum.photos/ a uložte ich do súborov:
```
https://picsum.photos/id/0/800/600
https://picsum.photos/id/1/800/600
https://picsum.photos/id/2/800/600
https://picsum.photos/id/3/800/600
https://picsum.photos/id/4/800/600
``` 

In [ ]:
// TODO: Doprogramujte


## Úloha 6

Prečítajte si tento článok: https://learn.microsoft.com/en-us/dotnet/csharp/asynchronous-programming/  

## Úloha 7
Vytvorte si nový projekt konzolovej aplikácie (cez Visual Studio 2022 / VS Code / Rider), ktorá bude sťahovať a zmenšovať obrázky pomocou NuGet knižnice `SixLabors.ImageSharp`. Vstupnými parametrami aplikácie bude veľkosť šírky a výšky, výstupný priečinok a ľubovoľný počet zadaných vstupných URL adries (ako napr. v 5. úlohe). Všetky takto zadané vstupné URL sa spracujú - obrázky sa stiahnu, zmenšia a uložia do súborov do výstupného priečinka.